In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_chimere_module import *
from association_module import *
from morbidity_chimere_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded all modules


In [2]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_sherpa(sc="s1", pol="ug_NO2", year=2019)

# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Concentrations in 2019 and 2019 are calculated for the pollutant 'ug_no2' (s1).
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod comname depcod depname  regcod           regname  \
0  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
1  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
2  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
3  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
4  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   

   age    pop2019    pop2030    pop2050  mort2019  mort2030  mort2050  \
0    0  31.979362  30.560224  28.974134  0.114652  0.101665  0.085457   
1    1  32.735555  30.384526  29.200275  0.018369  0.016510  0.014105   
2    2  33.511730  30.265005  29.417218  0.008333  0.007299  0.006310   
3    3  34.431724  30.206960  29.615683  0.005018  0.004605  0.004083   
4    4  35.587474  30.214303  29.787011  0.003853  0.003519  0.003135   

                                            geometry  
0  POLYGON ((497887

In [7]:
"""
This script:
- Runs CHIMERE -> correction -> expo for a future year (e.g., 2050)
- Aggregates IRIS (iriscod) exposure to commune (comcod) using population weights pop{year} from donnees_merged
- Builds morta_age dynamically from mort{year} (and pop_age from pop{year})
- Computes marginal avoided deaths + marginal years of life gained per 0.1 µg/m³ reduction
- Exports commune_meanconc_popw.csv (RAW meanconc; no WHO threshold applied there)
"""

import os
import logging
import warnings
import numpy as np
import pandas as pd

warnings.simplefilter(action="ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# --------------------------
# USER SETTINGS
# --------------------------
scenarios = ["s1", "s2", "s3", "s4"]   # "s1", "s2", "s3", "s4"
pollutants = ["ug_NO2"]   # add "ug_NO2" if needed
years = ["2030", "2050"]
delta_change = 0.1

# Use WHO thresholds here (set to 0.0 only if you want "total burden" not "excess above WHO")
RR_dict = {
    "ug_PM25_RH50": (1.10, 0.0),   # (RR per 10 µg/m³, WHO threshold)
    "ug_NO2":       (1.05, 0.0),
}
esp_dict = {2019: 80, 2030: 84, 2050: 86}

IRIS_COL = "iriscod"
COM_COL = "comcod"
AGE_COL = "age"  # if missing, years gained will be NaN

def _std_comcod(s: pd.Series) -> pd.Series:
    return s.astype(str).str.upper().str.zfill(5)

def _std_iriscod(s: pd.Series) -> pd.Series:
    return s.astype(str).str.upper().str.strip()

def infer_conc_col(donnees_expo: pd.DataFrame, pollutant: str) -> str:
    candidates = ["meanconc", "conc", "concentration", pollutant, pollutant.lower()]
    for c in candidates:
        if c in donnees_expo.columns:
            return c
    raise ValueError(
        f"Could not find concentration column in expo() output. "
        f"Tried {candidates}. Available columns: {list(donnees_expo.columns)}"
    )

def marginal_benefit_per_delta(df: pd.DataFrame, RR: float, delta: float, esp: float):
    """
    df must include: morta_age, meandelta; optionally age.
    """
    required = {"morta_age", "meandelta"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns for marginal calc: {missing}. Available: {list(df.columns)}")

    out = df.copy()

    # Avoided mortality at current meandelta
    out["avoided_mort"] = out["morta_age"] * (1 - np.exp(-np.log(RR) * out["meandelta"] / 10.0))

    # Avoided mortality after a marginal reduction by delta (clamp at 0)
    meandelta_minus = np.maximum(out["meandelta"] - float(delta), 0.0)
    out["avoided_mort_minus"] = out["morta_age"] * (1 - np.exp(-np.log(RR) * meandelta_minus / 10.0))

    # Benefit from delta reduction
    out["benefit_delta_deaths"] = out["avoided_mort"] - out["avoided_mort_minus"]
    total_deaths = float(out["benefit_delta_deaths"].sum())

    # Years of life gained (only if age is numeric years)
    if AGE_COL in out.columns:
        age_years = pd.to_numeric(out[AGE_COL], errors="coerce")
        remaining = np.maximum(float(esp) - age_years, 0.0)
        out["benefit_delta_years"] = out["benefit_delta_deaths"] * remaining
        total_years = float(out["benefit_delta_years"].sum(skipna=True))
    else:
        total_years = float("nan")

    return total_deaths, total_years, out


def aggregate_iris_to_commune_meanconc(
    donnees_expo: pd.DataFrame,
    donnees_merged: pd.DataFrame,
    year: str,
    pollutant: str,
) -> pd.DataFrame:
    """
    expo() output is IRIS-level: aggregate to commune using population weights pop{year} from donnees_merged.
    Returns DataFrame: comcod, meanconc (RAW concentration, no threshold applied).
    """
    df_expo = donnees_expo.copy()
    if IRIS_COL not in df_expo.columns:
        raise ValueError(f"expo() output must contain '{IRIS_COL}'. Available: {list(df_expo.columns)}")

    conc_col = infer_conc_col(df_expo, pollutant=pollutant)

    pop_col = f"pop{year}"
    if pop_col not in donnees_merged.columns:
        raise ValueError(
            f"donnees_merged must contain '{pop_col}' for population-weighting. "
            f"Available: {list(donnees_merged.columns)}"
        )

    # Minimal LUT with weights and commune codes at IRIS level
    lut = donnees_merged[[IRIS_COL, COM_COL, pop_col]].copy()
    lut[IRIS_COL] = _std_iriscod(lut[IRIS_COL])
    lut[COM_COL] = _std_comcod(lut[COM_COL])
    lut[pop_col] = pd.to_numeric(lut[pop_col], errors="coerce").fillna(0.0)

    df_expo[IRIS_COL] = _std_iriscod(df_expo[IRIS_COL])
    df_expo[conc_col] = pd.to_numeric(df_expo[conc_col], errors="coerce").fillna(0.0)

    df = df_expo.merge(lut, on=IRIS_COL, how="left")
    if df[COM_COL].isna().any():
        n = int(df[COM_COL].isna().sum())
        raise ValueError(f"{n} IRIS rows from expo() could not be mapped to comcod using donnees_merged.")

    def wmean(g):
        w = g[pop_col].to_numpy()
        x = g[conc_col].to_numpy()
        wsum = w.sum()
        if wsum <= 0:
            return float(np.mean(x))
        return float(np.average(x, weights=w))

    commune = (
        df.groupby(COM_COL, as_index=False)
          .apply(lambda g: pd.Series({"meanconc": wmean(g)}))
          .reset_index(drop=True)
          .rename(columns={COM_COL: "comcod"})
    )
    return commune[["comcod", "meanconc"]]


def build_commune_age_mortality(donnees_merged: pd.DataFrame, year: str) -> pd.DataFrame:
    """
    Build commune-age mortality dataframe with dynamic columns:
      morta_age <- mort{year}
      pop_age   <- pop{year}
    Returns: comcod, age (if present), morta_age, pop_age
    """
    #mort_col = f"mort{year}"
    #pop_col = f"pop{year}"
    mort_col = "mort2019"
    pop_col = "pop2019"

    needed = [IRIS_COL, COM_COL, mort_col, pop_col]
    missing = [c for c in needed if c not in donnees_merged.columns]
    if missing:
        raise ValueError(f"donnees_merged missing required columns: {missing}. Available: {list(donnees_merged.columns)}")

    cols = [COM_COL, mort_col, pop_col]
    if AGE_COL in donnees_merged.columns:
        cols.insert(1, AGE_COL)

    df = donnees_merged[cols].copy()
    df[COM_COL] = _std_comcod(df[COM_COL])
    df[mort_col] = pd.to_numeric(df[mort_col], errors="coerce").fillna(0.0)
    df[pop_col] = pd.to_numeric(df[pop_col], errors="coerce").fillna(0.0)

    df = df.rename(columns={mort_col: "morta_age", pop_col: "pop_age", COM_COL: "comcod"})

    group_cols = ["comcod"] + ([AGE_COL] if AGE_COL in df.columns else [])
    df = df.groupby(group_cols, as_index=False).agg({"morta_age": "sum", "pop_age": "sum"})
    return df


def process_combination(params, donnees_exportees_transformed, grille_combinee, donnees_merged,
                        delta=0.1, export_debug_csv=True):
    scenario, year, pollutant = params
    logging.info(f"Processing: Scenario={scenario}, Year={year}, Pollutant={pollutant}")

    if pollutant not in RR_dict:
        raise ValueError(f"Unknown pollutant '{pollutant}'. Add it to RR_dict.")

    output_path = os.path.join("data", "2-output-data", scenario, pollutant, year)
    os.makedirs(output_path, exist_ok=True)

    # --- Concentration pipeline (your functions) ---
    conc_points = coordo_chimere(pollutant, year=year, SC=scenario.upper())
    conc_points = conc_points.to_crs(donnees_exportees_transformed.crs)

    conc_chimere = coordo_ineris_chimere(pollutant, year="2019")
    conc_chimere = conc_chimere.to_crs(donnees_exportees_transformed.crs)

    conc_corrige = correction_chimere(conc_points, conc_chimere)

    donnees_expo = expo(donnees_exportees_transformed, conc_corrige, grille_combinee)
    logging.info("Expo processing completed successfully")

    # 1) Commune mean concentration (pop-weighted using pop{year} from donnees_merged)
    commune_conc = aggregate_iris_to_commune_meanconc(
        donnees_expo=donnees_expo,
        donnees_merged=donnees_merged,
        year=year,
        pollutant=pollutant,
    )

    # 2) Commune-age mortality (dynamic mort{year}, pop{year})
    mort_comm_age = build_commune_age_mortality(donnees_merged=donnees_merged, year=year)

    # 3) Merge conc with mortality
    df = mort_comm_age.merge(commune_conc, on="comcod", how="left")
    df["meanconc"] = df["meanconc"].fillna(0.0)

    RR, threshold = RR_dict[pollutant]
    df["meandelta"] = np.where(df["meanconc"] > threshold, df["meanconc"] - threshold, 0.0)

    # Restrict age range if age exists
    if AGE_COL in df.columns:
        df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")
        df = df[(df[AGE_COL] >= 30) & (df[AGE_COL] <= 99)].copy()

    esp = esp_dict.get(int(year), 80)

    total_deaths, total_years, df_out = marginal_benefit_per_delta(df, RR=RR, delta=delta, esp=esp)

    logging.info(f"[{scenario} {year} {pollutant}] Avoided deaths per {delta:.1f} µg/m³ reduction: {total_deaths:.2f}")
    logging.info(f"[{scenario} {year} {pollutant}] Years of life gained per {delta:.1f} µg/m³ reduction: {total_years:.2f}")

    if export_debug_csv:
        # RAW concentrations (no WHO threshold applied in this file)
        commune_conc.to_csv(os.path.join(output_path, "commune_meanconc_popw.csv"), index=False)
        #df_out.to_csv(os.path.join(output_path, f"marginal_{delta:.1f}ug_check.csv"), index=False)

    return total_deaths, total_years


if __name__ == "__main__":
    # Align CRS (your pattern)
    target_crs = donnees_exportees_transformed.crs
    donnees_exportees_transformed = donnees_exportees_transformed.to_crs(target_crs)
    grille_combinee = grille_combinee.to_crs(target_crs)

    params_list = [(sc, yr, pol) for sc in scenarios for yr in years for pol in pollutants]

    for params in params_list:
        process_combination(
            params,
            donnees_exportees_transformed=donnees_exportees_transformed,
            grille_combinee=grille_combinee,
            donnees_merged=donnees_merged,
            delta=delta_change,
            export_debug_csv=True,
        )

    logging.info("Successfully processed all combinations.")

INFO:root:Processing: Scenario=s1, Year=2030, Pollutant=ug_NO2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS1_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


INFO:root:Starting optimized expo function
INFO:root:Expo processing completed successfully
INFO:root:Expo processing completed successfully
INFO:root:[s1 2030 ug_NO2] Avoided deaths per 0.1 µg/m³ reduction: 364.54
INFO:root:[s1 2030 ug_NO2] Years of life gained per 0.1 µg/m³ reduction: 875.08
INFO:root:Processing: Scenario=s1, Year=2050, Pollutant=ug_NO2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS1_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


INFO:root:Starting optimized expo function
INFO:root:Expo processing completed successfully
INFO:root:Expo processing completed successfully
INFO:root:[s1 2050 ug_NO2] Avoided deaths per 0.1 µg/m³ reduction: 369.21
INFO:root:[s1 2050 ug_NO2] Years of life gained per 0.1 µg/m³ reduction: 1019.17
INFO:root:Processing: Scenario=s2, Year=2030, Pollutant=ug_NO2


...

INFO:root:Starting optimized expo function
INFO:root:Expo processing completed successfully
INFO:root:Expo processing completed successfully
INFO:root:[s3 2050 ug_NO2] Avoided deaths per 0.1 µg/m³ reduction: 368.47
INFO:root:[s3 2050 ug_NO2] Years of life gained per 0.1 µg/m³ reduction: 1017.09
INFO:root:Processing: Scenario=s4, Year=2030, Pollutant=ug_NO2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS4_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


INFO:root:Starting optimized expo function
INFO:root:Expo processing completed successfully
INFO:root:Expo processing completed successfully
INFO:root:[s4 2030 ug_NO2] Avoided deaths per 0.1 µg/m³ reduction: 362.87
INFO:root:[s4 2030 ug_NO2] Years of life gained per 0.1 µg/m³ reduction: 870.98
INFO:root:Processing: Scenario=s4, Year=2050, Pollutant=ug_NO2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS4_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


INFO:root:Starting optimized expo function
INFO:root:Expo processing completed successfully
INFO:root:Expo processing completed successfully
INFO:root:[s4 2050 ug_NO2] Avoided deaths per 0.1 µg/m³ reduction: 368.24
INFO:root:[s4 2050 ug_NO2] Years of life gained per 0.1 µg/m³ reduction: 1016.45
INFO:root:Successfully processed all combinations.
